# Resume Screening & Skill Matching - End-to-End Demo

**What this notebook demonstrates**
1. Clone the repo + install dependencies
2. **Core matching** - rank resumes against a job description (SBERT semantic + skill overlap)
3. **Explainability** - matched/missing skills with evidence + structured experience/education + upskilling roadmap
4. **Multilingual** - an English JD matching Spanish/French resumes cross-lingually
5. **Active learning** - recruiter feedback -> trained re-ranker -> adaptive scoring (with interpretable coefficients)
6. **Learning curve** - metrics improving as feedback accumulates
7. **Benchmark metrics** - Precision / Recall / F1 / MRR / NDCG / cosine separation / timing
8. **Robustness** - ranking stability on truncated (incomplete) resumes
9. **API interface** - the FastAPI endpoints the system exposes


## 1. Setup - clone the repo and install dependencies

In [ ]:
!git clone https://github.com/devn-cmd/Resume-Matcher.git
%cd Resume-Matcher

In [ ]:
!pip install -q sentence-transformers spacy langdetect scikit-learn pandas pdfplumber python-docx matplotlib fastapi

In [ ]:
import sys
sys.path.insert(0, '.')
print("Eval data:")
!ls data/eval
print("\nDemo resumes:")
!ls data/raw/resumes

## 2. Core matching 
-rank resumes against a job description

Loads the sample job description and the 10 demo resumes, then ranks them by fit.

In [ ]:
from pathlib import Path
import pandas as pd
from src.matching import ResumeMatcher
from src.ingestion import read_document

# Sample job description (Data Scientist)
jd = Path('data/raw/jd_data_scientist.txt').read_text(encoding='utf-8')

# Read every resume in data/raw/resumes (PDF / DOCX / TXT)
resume_dir = Path('data/raw/resumes')
resumes = {f.name: read_document(str(f)) for f in sorted(resume_dir.iterdir()) if f.is_file()}

matcher = ResumeMatcher()
results = matcher.rank(jd, resumes)

pd.DataFrame([{
    'Candidate':      r.resume_id,
    'Language':       r.language,
    'Cross-lingual':  r.cross_lingual,
    'Final score':    r.final_score,
    'Semantic':       r.semantic_score,
    'Skill match':    r.skill_score,
    'Suitable':       matcher.is_suitable(r),
} for r in results])

## 3. Explainability - why did the top candidate match?

Shows the matched skills with the exact resume sentence as evidence, the **structured experience/education** extracted from the resume, and the personalized upskilling roadmap for any missing skills.

In [ ]:
import config
from src.skills import SkillExtractor
from src.explain import explain

extractor = SkillExtractor(config.SKILLS_DB_PATH)
top = results[0]
info = explain(resumes[top.resume_id], set(top.matched_skills), set(top.missing_skills), extractor)

print(f"TOP CANDIDATE: {top.resume_id}\n")
print("Matched skills (with evidence):")
for skill, sentence in info['evidence'].items():
    print(f"  - {skill}: {sentence}")

print(f"\nMissing skills: {sorted(top.missing_skills) or '-'}")

# Structured entity extraction (spec: extract experience / education / certifications)
struct = info['structured']
if getattr(struct, 'experience', None):
    print("\nExperience (extracted):")
    for e in struct.experience[:3]:
        print(f"  - {e.raw}")
if getattr(struct, 'education', None):
    print("\nEducation (extracted):")
    for e in struct.education[:3]:
        print(f"  - {e.raw}")

if info['recommendations']:
    print("\nUpskilling roadmap for missing skills:")
    for rec in info['recommendations']:
        print(f"  - {rec['skill']} (market demand {rec['demand_score']})")
        for step in rec['roadmap']:
            print(f"      Step {step['step']}: {step['action']}")

## 4. Multilingual - cross-lingual matching

The English job description is matched directly against resumes written in other languages, with no translation step.

In [ ]:
cross = [r for r in results if r.cross_lingual]
if cross:
    print("Resumes matched cross-lingually (different language from the JD):\n")
    for r in cross:
        print(f"  - {r.resume_id}  -  detected {r.language}, "
              f"final score {r.final_score:.2f}, suitable={matcher.is_suitable(r)}")
else:
    print("No cross-lingual resumes in this set.")

## 5. Active learning - the recruiter feedback loop

The system learns from recruiter shortlist/reject decisions. A small, **interpretable** logistic-regression layer (a classification layer) is trained on accumulated feedback and re-scores candidates. The coefficients *are* the explanation - you can read exactly what drives each decision.

Privacy: feedback is stored as feature vectors + a label only (no raw resume text), satisfying the spec's data-privacy constraint.

In [ ]:
from src.feedback import load_feedback, count_feedback, class_balance
from src.active_learning import ActiveLearningRanker

pos, neg = class_balance()
print(f"Feedback collected: {count_feedback()}  ({pos} shortlists / {neg} rejects)\n")

# Train the adaptive re-ranker on the accumulated feedback
ranker = ActiveLearningRanker()
report = ranker.fit(load_feedback())
print(f"Re-ranker trained: {report.trained}  (on {report.n_samples} samples)\n")

print("Learned coefficients (interpretability - positive pushes toward 'suitable'):")
for feature, weight in ranker.coefficients().items():
    print(f"  {feature:>18}: {weight:+.3f}")

Now re-rank the same candidates **with** the trained model. Compare the static blend against the adaptive score, and note which candidates the model flags as *uncertain* (its review queue).

In [ ]:
adaptive = matcher.rank_with_feedback(jd, resumes, ranker=ranker)

pd.DataFrame([{
    'Candidate':  r.resume_id,
    'Static':     r.final_score,
    'Adaptive':   round(r.adaptive_score, 4) if r.adaptive_score is not None else None,
    'In review queue': r.is_uncertain,
    'Suitable':   matcher.is_suitable(r),
} for r in adaptive])

## 6. Learning curve - does feedback actually improve the model?

First load the evaluation set, then show the saved learning curve. The curve plots metrics on a held-out test slice against the number of recruiter labels collected.

In [ ]:
from evaluation.eval_data import load_eval_data
jd_texts, eval_resumes, labels = load_eval_data()

In [ ]:
from evaluation.active_learning_eval import plot_curve
from IPython.display import Image

curve = pd.read_csv('evaluation/active_learning_curve.csv')
display(curve)
plot_curve(curve)
Image('evaluation/active_learning_curve.png')

## 7. Benchmark metrics

The standard evaluation: Precision, Recall, F1, MRR, NDCG, cosine separation between matched/unmatched pairs, and processing time per resume.

In [ ]:
from evaluation.evaluate import evaluate
metrics = evaluate(jd_texts, eval_resumes, labels)
metrics

## 8. Robustness - incomplete resumes

The spec asks for robustness against noisy/incomplete data. Here we truncate each resume to its first 30% and measure how much the ranking shifts. Low drift = the ranking holds up when resumes are incomplete.

In [ ]:
import numpy as np

baseline_order = [r.resume_id for r in matcher.rank(jd, resumes)]
truncated = {rid: text[:max(1, int(len(text) * 0.3))] for rid, text in resumes.items()}
trunc_order = [r.resume_id for r in matcher.rank(jd, truncated)]

drift = np.mean([abs(baseline_order.index(rid) - trunc_order.index(rid)) for rid in resumes])
print(f"Average rank drift when resumes are truncated to 30%: {drift:.2f} positions")
print("(Lower is better - the ranking is robust to incomplete resumes.)")

## 9. API interface

The system also exposes a FastAPI service (`api/main.py`) for programmatic integration. Listing its routes confirms the endpoints exist (no server needed).

In [ ]:
from api.main import app

print("FastAPI endpoints exposed by api/main.py:\n")
for route in app.routes:
    methods = ",".join(sorted(route.methods)) if getattr(route, 'methods', None) else ""
    if route.path.startswith('/') and not route.path.startswith('/openapi'):
        print(f"  {methods:12} {route.path}")

## 10.Launch the interactive dashboard

launching it  through a public tunnel, but tunnels can be flaky
## So i add streamlit dashboard link in  README.md

In [ ]:
!pip install -q streamlit
!streamlit run dashboard/app.py &>./streamlit_log.txt &
import time; time.sleep(8)
import urllib.request
print("Tunnel password (your Colab public IP):",
      urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode())
!npx --yes localtunnel --port 8501